# 🧠 Module 2: Embeddings & Vector Search

---

## 🎯 Objective
Learn how to convert text into embeddings and perform similarity search.

## 🧠 Key Learnings

- Embeddings = meaning in vector form
- Similar meaning → closer vectors
- FAISS = fast similarity search
- This replaces keyword search


## ⚙️ Setup

### ⚙️ Step 1: Install Dependencies

```bash
pip install sentence-transformers faiss-cpu numpy
```

### ⚙️ Step 2: Setup Code

In [2]:
from sentence_transformers import SentenceTransformer # Used to convert text into embeddings (vectors)
import numpy as np # For numerical operations
import faiss # Facebook AI Similarity Search (fast vector search)

# Load a pre-trained embedding model
# This model converts sentences into numerical vectors (embeddings)
model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/abhishekjha/Documents/ai-learning/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 🧪 Experiment 1: Generate Embeddings (Text → Embeddings)

---
**💡 Insight**
- Each sentence → high-dimensional vector
- Similar meaning → similar vectors

In [3]:
sentences = [
    "I love programming",
    "I enjoy coding",
    "The sky is blue"
]
# Convert sentences into embeddings (vectors)
embeddings = model.encode(sentences)

# (3 sentences, each converted into a vector of fixed size e.g. 384)
print("Shape:", embeddings.shape)
print("First vector:", embeddings[0][:5])

Shape: (3, 384)
First vector: [-0.03617872 -0.0127737   0.00300632 -0.01690342  0.00948422]


## 🧪 Experiment 2: Similarity Check

---

**💡 Expected**
Programming ≈ Coding → high similarity
Programming ≠ Sky → low similarity

In [4]:
# Import norm (used for cosine similarity calculation)
from numpy.linalg import norm

# Function to calculate cosine similarity between two vectors
def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("Programming vs Coding:",
      cosine_similarity(embeddings[0], embeddings[1]))

print("Programming vs Sky:",
      cosine_similarity(embeddings[0], embeddings[2]))

Programming vs Coding: 0.81720906
Programming vs Sky: 0.08906172


## 🧪 Experiment 3: Build Simple Search Engine (FAISS Vector Search)

---

Building a semantic search engine

#### Step 1: Create Index

In [5]:
# Get vector dimension (e.g., 384)
dimension = embeddings.shape[1]

# Create FAISS index using L2 distance (Euclidean distance)
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index (store vectors)
index.add(embeddings)

#### Step 2: Query Search

In [6]:
# Define a query
query = "I like writing code"

# Convert query into embedding
query_vector = model.encode([query])

# Search top 2 most similar vectors
# distances: how far results are
# indices: positions of matching sentences
distances, indices = index.search(query_vector, k=2)

print("Top matches:")

# Retrieve actual sentences using indices
for i in indices[0]:
    print(sentences[i])

Top matches:
I enjoy coding
I love programming


## 🏗 Mini Project: Semantic Search System

#### Step 1: Documents

In [7]:
# List of documents (can be anything: notes, FAQs, etc.)
documents = [
    "Python is a programming language",
    "Cats are animals",
    "Dogs are loyal pets",
    "Java is used in enterprise applications",
    "Machine learning is a subset of AI"
]

#### Step 2: Embed + Store

In [8]:
# Convert all documents into embeddings
doc_embeddings = model.encode(documents)

# Create FAISS index
index = faiss.IndexFlatL2(doc_embeddings.shape[1])

# Store document embeddings in index
index.add(doc_embeddings)

#### Step 3: Search Function

In [9]:
def search(query, k=2):
    """
    Perform semantic search
    
    Args:
        query (str): user input
        k (int): number of results to return
    
    Returns:
        list: top matching documents
    """
    
    # Convert query to embedding
    query_vector = model.encode([query])
    
    # Search similar vectors
    distances, indices = index.search(query_vector, k)
    
    # Return matching documents
    return [documents[i] for i in indices[0]]

#### Step 4: Test

In [11]:
# Test different queries

print("Query: What is AI?")
print(search("What is AI?"))

print("\nQuery: Programming languages")
print(search("Programming languages"))

print("\nQuery: Animals")
print(search("Animals"))

# You may get unexpected results
print("\nQuery: Tell me about loyalty")
print(search("Tell me about loyalty"))

Query: What is AI?
['Machine learning is a subset of AI', 'Python is a programming language']

Query: Programming languages
['Python is a programming language', 'Java is used in enterprise applications']

Query: Animals
['Cats are animals', 'Dogs are loyal pets']

Query: Tell me about loyalty
['Dogs are loyal pets', 'Cats are animals']
